# Hamiltonian from SKQD paper

In [ ]:
import numpy as np

from qiskit.quantum_info import SparsePauliOp
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper


def siam_diagonal_bath_sparse_pauli(
    K: int,
    U: float,
    t: float = 1.0,
    V: float = 1.0,
):
    """
    Single-impurity Anderson model in the bath-diagonal basis.

    Modes per spin sector:
      0        -> impurity orbital d
      1..K     -> bath orbitals k=0..K-1

    Total spin orbitals / qubits = 2 * (K + 1), with ordering:
      [up impurity, up bath0, ..., up bath_{K-1},
       dn impurity, dn bath0, ..., dn bath_{K-1}]

    Returns
    -------
    ham_pauli : SparsePauliOp
        Jordan-Wigner mapped qubit Hamiltonian.
    eps : np.ndarray
        Bath single-particle energies.
    Vk : np.ndarray
        Hybridizations in the diagonal-bath basis.
    """
    n_orb = K + 1                    # spatial orbitals per spin
    n_spin_orb = 2 * n_orb

    # Open-chain bath hopping matrix T, Eq. (19)
    T = np.zeros((K, K), dtype=float)
    for j in range(K - 1):
        T[j, j + 1] = -t
        T[j + 1, j] = -t

    # Diagonalize bath: eps_k and Xi
    eps, Xi = np.linalg.eigh(T)
    Vk = V * Xi[0, :]                # Eq. (21)

    def idx(spin: str, orb: int) -> int:
        # orb = 0 is impurity, 1..K are bath orbitals
        if spin == "up":
            return orb
        elif spin == "dn":
            return n_orb + orb
        raise ValueError("spin must be 'up' or 'dn'")

    terms = {}

    def add_term(label: str, coeff: complex):
        terms[label] = terms.get(label, 0.0) + coeff

    # Impurity one-body term: U/2 (n_d↑ + n_d↓), Eq. (15)
    add_term(f"+_{idx('up', 0)} -_{idx('up', 0)}", U / 2)
    add_term(f"+_{idx('dn', 0)} -_{idx('dn', 0)}", U / 2)

    # Impurity interaction: U n_d↑ n_d↓, Eq. (15)
    add_term(
        f"+_{idx('up', 0)} -_{idx('up', 0)} +_{idx('dn', 0)} -_{idx('dn', 0)}",
        U,
    )

    # Bath diagonal term: sum_{k,sigma} eps_k n_{k sigma}, Eq. (20)
    for k in range(K):
        add_term(f"+_{idx('up', k + 1)} -_{idx('up', k + 1)}", eps[k])
        add_term(f"+_{idx('dn', k + 1)} -_{idx('dn', k + 1)}", eps[k])

    # Hybridization: sum_{k,sigma} Vk (d† c_k + c_k† d), Eq. (21)
    for k in range(K):
        vk = Vk[k]
        # spin up
        add_term(f"+_{idx('up', 0)} -_{idx('up', k + 1)}", vk)
        add_term(f"+_{idx('up', k + 1)} -_{idx('up', 0)}", vk)
        # spin down
        add_term(f"+_{idx('dn', 0)} -_{idx('dn', k + 1)}", vk)
        add_term(f"+_{idx('dn', k + 1)} -_{idx('dn', 0)}", vk)

    ferm_op = FermionicOp(terms, num_spin_orbitals=n_spin_orb)
    mapper = JordanWignerMapper()
    ham_pauli = mapper.map(ferm_op).simplify()

    return ham_pauli, eps, Vk